In [1]:
import json
import random
import base64
import os
import shutil
import subprocess
import dill
import nltk
import config
import itertools
import io
import taxonomy

from pydantic import BaseModel
from utils.DataHandlers import LatexDataHandler
from utils.ModelClient import ModelClient
from pathlib import Path
from collections import Counter
from pdf2image import convert_from_bytes

In [ ]:
SYSTEM_PROMPT = """# Identify
You are a judge that scores the quality of LLM generated critiques of scientific manuscripts. 

# Task
An LLM was prompted to critique the quality of a scientific manuscript after specific types of table-text weaknesses were introduced into the manuscript. Your task is to judge whether the LLM response correctly identifies the weaknesses that were introduced.

You will be provided with:
* The latex of a scientific manuscript.
* The table-text weakness type that was introduced into the manuscript and its description.
* A list of edits that were performed on tables to introduce the weakness into the manuscript. An edit consists of two parts: a unique span of text from the original manuscript, and a second span of text that replaces the first.
* A list of sentences from the manuscript that demonstrate the existence of the weakness in the manuscript after the edits are performed.
* An explanation of why the edit introduces the specified type of weakness into the manuscript.
* A model response critiquing the manuscript.

Your task is to score the provided model response using the rubric provided below.

# Rubric
Use the following rubric to score the quality of the model response.

0: The response does not identify any weaknesses in the manuscript that are relevant to the introduced edits.
1: The response identifies one or more weaknesses that are relevant to the introduced edits, but the explanation of these weaknesses does not match the introduced weakness type.
2: The response identifies a weakness of the same type, but it is not clear whether it is the same weakness that was introduced into the manuscript.
3: The response identifies the specific weakness that was introduced into the manuscript, but the response is not detailed enough to localize the weakness to sentences or tables in the manuscript.
4: The response identifies the specific weakness that was introduced into the manuscript and provides sufficient detail to localize it to specific sentences or tables in the manuscript.
"""

MESSAGE_TEMPLATE = """
Manuscript Latex:
{0}
Weakness Type: 
{1}
Weakness Description: 
{2}
Edits Introduced:
{3}
Sentences Demonstrating Weakness:
{4}
Explanation of Edits:
{5}
Model Response:
{6}
"""



In [ ]:
class Score(BaseModel):
    score: int

In [4]:
def judge_iterator(model_responses):
    for paper_id, error_id, res in model_responses:
        # no responses - skip
        if 'model_responses' not in res.keys() or len(res['model_responses']) < 1:
            continue

        meta_id = f'{paper_id}___{error_id}'

        latex_path = DATA_FOLDER_PATH / paper_id / 'flat_main.tex'
        with open(latex_path, 'r') as fp:
            manuscript_latex = fp.read()
        
        weakness_type = base64.urlsafe_b64decode(error_id).decode()
        weakness_description = taxonomy.TAXONOMY_DATA[weakness_type]
        sentences = res['sentences']
        edit_list = res['edit_list']
        edit_reasoning = res['reasoning']

        for model_response in res['model_responses']:
            model_id = model_response['model_id']
            meta_id = f'{paper_id}___{error_id}___{model_id}'
            message = MESSAGE_TEMPLATE.format(
                manuscript_latex,
                weakness_type,
                weakness_description,
                edit_list,
                sentences,
                edit_reasoning,
                model_response['response_text']
            )

            req_data = {
                'system_prompt': SYSTEM_PROMPT,
                'message': message,
                'structured_output_format': Score,
                'reasoning_effort': 'high',
            }

            yield meta_id, req_data

In [5]:
DATA_FOLDER_PATH = config.DATA_FOLDER_PATH

In [6]:
JUDGE_MODEL_INFO = config.JUDGE_MODEL_INFO
model_client = ModelClient(*JUDGE_MODEL_INFO)

In [7]:
with open(f'{config.OUTPUT_FOLDER_PATH}model_responses.dill', 'rb') as fp:
    model_responses = dill.load(fp)

In [8]:
cache_path = f'{config.OUTPUT_FOLDER_PATH}responses_judge_cache'
for meta_id, res in model_client.iterative_query(judge_iterator(model_responses)):
    print(meta_id)
    paper_id, error_id, model_id = meta_id.split('___')

    for model_response in model_responses:
        if model_response[0] == paper_id and model_response[1] == error_id:
            for mr in model_response[2]['model_responses']:
                if mr['model_id'] == model_id:
                    mr['judgement'] = res
            break

with open(f'{config.OUTPUT_FOLDER_PATH}responses_judge.dill', 'wb') as fp:
    dill.dump(model_responses, fp)

2409.14664___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_52_mini
2409.14664___bG9naWNhbGx5X2ludmFsaWRfaW50ZXJwcmV0YXRpb24=___gpt_52_mini
2409.14664___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=___gpt_52_mini
2409.14664___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___gpt_52_mini
2409.14664___Y29udHJhZGljdGlvbl93aXRoX2VzdGFibGlzaGVkX2tub3dsZWRnZQ==___gpt_52_mini
2502.10118___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___gpt_52_mini
2502.10118___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=___gpt_52_mini
2502.10118___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___gpt_52_mini
2502.10118___Y29udHJhZGljdGlvbl93aXRoX2VzdGFibGlzaGVkX2tub3dsZWRnZQ==___gpt_52_mini


/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot locate reference to <class '__main__.Score'>.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot pickle <class '__main__.Score'>: __main__.Score has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


In [9]:
with open(f'{config.OUTPUT_FOLDER_PATH}responses_judge.dill', 'rb') as fp:
    model_responses_all = dill.load(fp)

In [10]:
for meta_id, error_id, res in model_responses_all:
    if 'model_responses' not in res:
        continue

    for mr in res['model_responses']:
        #print(mr['response_text'])
        print(mr['judgement'])
        print('-----')
    print('=====')

score=0
-----
=====
score=1
-----
=====
score=0
-----
=====
score=0
-----
=====
score=0
-----
=====
score=0
-----
=====
score=4
-----
=====
score=1
-----
=====
score=4
-----
=====
